In [14]:
import pandas as pd
import numpy as np
 
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    f1_score,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt
import joblib

In [15]:
# Load Dataset
df = pd.read_csv("../data/engineered/deforestation_training.csv")

# Features and Target
X = df.drop("Deforestation_Risk", axis=1)
y = df["Deforestation_Risk"]

In [16]:
# Data Splitting(70% Train, 15% Validation 15% Testing)
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (975, 25)
Validation: (209, 25)
Testing: (209, 25)


In [17]:
# Hyperparameter Tuning with Cross-Validation
# (search happens ONLY on training data; val/test remain untouched)
param_dist = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [5, 10, 15, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "class_weight": ["balanced", "balanced_subsample", None],
}
 
base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
 
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=40,
    scoring="f1_macro",
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True,
)
 
print("\nRunning cross-validated hyperparameter search...")
search.fit(X_train, y_train)
 
print("\nBest Parameters Found:")
print(search.best_params_)
print(f"Best Cross-Validation Macro F1: {search.best_score_:.4f}")
 
model = search.best_estimator_


Running cross-validated hyperparameter search...
Fitting 5 folds for each of 40 candidates, totalling 200 fits

Best Parameters Found:
{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20, 'class_weight': 'balanced_subsample'}
Best Cross-Validation Macro F1: 0.9958


In [18]:
# Validation
val_pred = model.predict(X_val)

print("\nValidation Accuracy:", accuracy_score(y_val, val_pred))
print("Validation Macro F1 :", f1_score(y_val, val_pred, average="macro"))

# Test Evaluation
test_pred = model.predict(X_test)
 
accuracy = accuracy_score(y_test, test_pred)
precision = precision_score(y_test, test_pred, average="weighted")
recall = recall_score(y_test, test_pred, average="weighted")
f1 = f1_score(y_test, test_pred, average="weighted")
macro_f1 = f1_score(y_test, test_pred, average="macro")
 
print("\nFinal Test Results")
print("---------------------")
print("Accuracy        :", accuracy)
print("Precision       :", precision)
print("Recall          :", recall)
print("F1 Score (wtd)  :", f1)
print("F1 Score (macro):", macro_f1)
 
print("\nClassification Report")
print(classification_report(y_test, test_pred))
 
print("\nConfusion Matrix")
print(confusion_matrix(y_test, test_pred))

# Feature Importance
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})
 
importance = importance.sort_values(
    by="Importance",
    ascending=False
)
 
print("\nTop 10 Important Features")
print(importance.head(10))
 
# Save Feature Importance
importance.to_csv(
    "../src/feature_importance.csv",
    index=False
)


Validation Accuracy: 1.0
Validation Macro F1 : 1.0

Final Test Results
---------------------
Accuracy        : 0.9952153110047847
Precision       : 0.9952641343618787
Recall          : 0.9952153110047847
F1 Score (wtd)  : 0.9952071320492374
F1 Score (macro): 0.9954415954415955

Classification Report
              precision    recall  f1-score   support

           0       0.99      1.00      0.99        97
           1       1.00      1.00      1.00        53
           2       1.00      0.98      0.99        59

    accuracy                           1.00       209
   macro avg       1.00      0.99      1.00       209
weighted avg       1.00      1.00      1.00       209


Confusion Matrix
[[97  0  0]
 [ 0 53  0]
 [ 1  0 58]]

Top 10 Important Features
                       Feature  Importance
1           Tree_Cover_Loss_ha    0.432970
4               Fire_Incidents    0.124740
5         Illegal_Logging_Flag    0.117572
0               Forest_Area_ha    0.115630
2           Annual_R

In [19]:
# Save Metrics
with open("../src/metrics.txt", "w") as f:
    f.write("Best Hyperparameters:\n")
    f.write(f"{search.best_params_}\n\n")
    f.write(f"Best CV Macro F1: {search.best_score_:.4f}\n\n")
    f.write(f"Validation Accuracy: {accuracy_score(y_val, val_pred)}\n")
    f.write(f"Validation Macro F1: {f1_score(y_val, val_pred, average='macro')}\n\n")
    f.write(f"Accuracy : {accuracy}\n")
    f.write(f"Precision: {precision}\n")
    f.write(f"Recall   : {recall}\n")
    f.write(f"F1 Score (weighted): {f1}\n")
    f.write(f"F1 Score (macro)   : {macro_f1}\n\n")
    f.write(classification_report(y_test, test_pred))

# Save Cross-Validation Search Results
cv_results = pd.DataFrame(search.cv_results_).sort_values(
    "mean_test_score", ascending=False
)
cv_results.to_csv("../src/cv_search_results.csv", index=False)

# Save Model
joblib.dump(model, "../src/random_forest_model.pkl")

print("\nModel Saved Successfully!")


Model Saved Successfully!
